# The Transformer by hand: inference

The last notebook trained-style: the whole German sentence went in at once. This one does what a finished model does with a sentence it has never seen: it **writes the translation itself, one token at a time**, feeding every word it picks back in, until it writes `</s>`.

It uses the same tiny model as the encoder + decoder notebook ("the cat sat", d_model 4, 2 heads, a vocabulary of 11). You don't need to redo the layers: the decoder is given to you as a function, `last_logits(tokens)`, built from exactly the pieces you filled in last time. Your job is the part that's new.

**What you'll do**

1. **Greedy decoding.** Each turn, turn the last row's logits into probabilities, pick the most likely word, and feed it back in. Repeat until `</s>` (or 4 turns).
2. **The KV cache.** Push one new token through one self-attention head using cached keys and values, and check it gives the same row as recomputing everything.
3. **Beam search.** Keep the 2 best partial translations for two turns, and compute the length penalty.

**How it works:** run the three setup cells. Each step then has a cell full of `nan`: replace them with your numbers and run it. Wrong entries are listed with a hint; `reveal('name')` shows an answer and lets you carry on. Use Python as a calculator for single numbers (`np.exp(1.3)`, `np.log(0.4)`), but not for whole rows at once. Two or three decimals are enough.

| setting | here | paper |
|---|---|---|
| source sentence | "the cat sat" (n = 3) | any |
| d_model, heads, d_k | 4, 2, 2 | 512, 8, 64 |
| layers | 1 encoder + 1 decoder | 6 + 6 |
| vocabulary V | 11 | ≈ 37,000 |
| maximum output length | 4 | n + 50 |
| beam size, length penalty α | 2, 0.6 | 4, 0.6 |

The weights are made-up stand-ins, not trained ones, so the "translation" won't be German. The loop is exactly the one a trained model runs.

## Setup

Run these three cells in order. The first two are the same as in the encoder + decoder notebook.

In [ ]:
# Run this cell first. It sets up the example, the weights and the checker.
# You don't need to read it, and please don't peek at REF: that's the answer key.
import numpy as np
np.set_printoptions(precision=4, suppress=True)
nan = np.nan

# ---- settings for this run (paper values in comments) ----
sentence = "the cat sat"
n       = 3    # tokens            (any length)
d_model = 4    # vector width      (paper: 512)
h       = 2    # heads             (paper: 8)
d_k     = d_model // h   # = 2     (paper: 64)
d_ff    = 8    # FFN inner width   (paper: 2048)

# ---- a tiny vocabulary and its (made-up) learned embedding table, V x d_model ----
vocab = {"<pad>": 0, "the": 1, "cat": 2, "sat": 3, "on": 4, "mat": 5}
W_emb = np.array([
    [ 0.0,  0.0,  0.0,  0.0],   # 0 <pad>
    [ 0.1,  0.2, -0.1,  0.3],   # 1 the
    [ 0.4, -0.2,  0.3,  0.1],   # 2 cat
    [-0.3,  0.5,  0.2, -0.1],   # 3 sat
    [ 0.2,  0.1, -0.4,  0.2],   # 4 on
    [ 0.3, -0.1,  0.1,  0.4],   # 5 mat
])

# ---- attention weights (made up; a trained model learns these), each d_model x d_k ----
W_Q1 = np.array([[ 1.0, 0.0], [0.0,  1.0], [ 0.5, 0.0], [0.0, -0.5]])
W_K1 = np.array([[ 0.0, 1.0], [1.0,  0.0], [ 0.0, 0.5], [-0.5, 0.0]])
W_V1 = np.array([[ 1.0, 0.0], [0.0,  0.0], [ 0.0, 1.0], [ 0.5, 0.5]])
W_Q2 = np.array([[ 0.0, 0.5], [1.0,  0.0], [ 0.0, 1.0], [ 0.5, 0.0]])
W_K2 = np.array([[ 0.5, 0.0], [0.0,  1.0], [ 1.0, 0.0], [ 0.0, 0.5]])
W_V2 = np.array([[ 0.0, 1.0], [0.5,  0.0], [ 1.0, 0.0], [ 0.0, -0.5]])
W_O  = np.array([[ 0.5, 0.0, 0.0, 0.5],
                 [ 0.0, 0.5, 0.5, 0.0],
                 [ 0.0, 0.5, -0.5, 0.0],
                 [ 0.5, 0.0, 0.0, -0.5]])

# ---- feed-forward weights: W1 is d_model x d_ff, W2 is d_ff x d_model ----
W1 = np.array([[ 0.5, -0.5,  0.0,  1.0,  0.0, -1.0,  0.5,  0.0],
               [ 0.0,  0.5,  0.5,  0.0, -0.5,  0.0,  0.0,  1.0],
               [ 1.0,  0.0, -0.5,  0.0,  0.5,  0.5,  0.0, -0.5],
               [ 0.0,  0.0,  0.5, -0.5,  0.0,  0.0,  1.0,  0.0]])
b1 = np.array([0.0, 0.1, 0.0, -0.1, 0.0, 0.1, 0.0, -0.1])
W2 = np.array([[ 0.5,  0.0,  0.0,  0.0],
               [ 0.0,  0.5,  0.0,  0.0],
               [ 0.0,  0.0,  0.5,  0.0],
               [ 0.0,  0.0,  0.0,  0.5],
               [-0.5,  0.0,  0.5,  0.0],
               [ 0.0, -0.5,  0.0,  0.5],
               [ 0.5,  0.5,  0.0,  0.0],
               [ 0.0,  0.0, -0.5, -0.5]])
b2 = np.array([0.1, 0.0, -0.1, 0.0])
eps = 1e-5   # LayerNorm's tiny safety constant; it won't change your 2-decimal answers

# =============== the checker (no need to read below) ===============
def _sm(S):
    e = np.exp(S - S.max(axis=1, keepdims=True)); return e / e.sum(axis=1, keepdims=True)
def _ln(R):
    mu = R.mean(axis=1); sd = np.sqrt(R.var(axis=1) + eps)
    return mu, sd, (R - mu[:, None]) / sd[:, None]

_ok = {}
def got(k): return _ok[k] if k in _ok else REF[k]

_RULES = {   # how each matrix is computed from the matrices before it
  "ids":  lambda: np.array([vocab[t] for t in sentence.split()], dtype=float),
  "E":    lambda: W_emb[got("ids").astype(int)],
  "Es":   lambda: got("E") * np.sqrt(d_model),
  "div":  lambda: np.array([10000 ** (2 * i / d_model) for i in range(d_model // 2)]),
  "PE":   lambda: np.array([[f(p / got("div")[c // 2]) for c, f in
                             zip(range(d_model), [np.sin, np.cos] * (d_model // 2))] for p in range(n)]),
  "X":    lambda: got("Es") + got("PE"),
  "Q1":   lambda: got("X") @ W_Q1, "K1": lambda: got("X") @ W_K1, "V1": lambda: got("X") @ W_V1,
  "Q2":   lambda: got("X") @ W_Q2, "K2": lambda: got("X") @ W_K2, "V2": lambda: got("X") @ W_V2,
  "S1":   lambda: got("Q1") @ got("K1").T, "S2": lambda: got("Q2") @ got("K2").T,
  "S1s":  lambda: got("S1") / np.sqrt(d_k), "S2s": lambda: got("S2") / np.sqrt(d_k),
  "A1":   lambda: _sm(got("S1s")), "A2": lambda: _sm(got("S2s")),
  "H1":   lambda: got("A1") @ got("V1"), "H2": lambda: got("A2") @ got("V2"),
  "C":    lambda: np.concatenate([got("H1"), got("H2")], axis=1),
  "MHA":  lambda: got("C") @ W_O,
  "R1":   lambda: got("X") + got("MHA"),
  "mu1":  lambda: _ln(got("R1"))[0], "sd1": lambda: _ln(got("R1"))[1],
  "X1":   lambda: (got("R1") - got("mu1")[:, None]) / got("sd1")[:, None],
  "P":    lambda: got("X1") @ W1 + b1,
  "Hf":   lambda: np.maximum(0, got("P")),
  "F":    lambda: got("Hf") @ W2 + b2,
  "R2":   lambda: got("X1") + got("F"),
  "mu2":  lambda: _ln(got("R2"))[0], "sd2": lambda: _ln(got("R2"))[1],
  "Z":    lambda: (got("R2") - got("mu2")[:, None]) / got("sd2")[:, None],
}
REF = {}
for _k, _f in _RULES.items():   # the full answer key, computed in order
    REF[_k] = _f()

_HINTS = {
  "ids": "Look each word up in the vocab dictionary.",
  "E": "Row r is the row of W_emb whose number is ids[r]. No multiplication, just copy the row.",
  "Es": "Multiply every entry of E by sqrt(d_model) = sqrt(4) = 2.",
  "div": "Pair i uses 10000 ** (2*i / d_model). With d_model = 4 that's 10000**0 and 10000**0.5.",
  "PE": "Even column 2i: sin(pos / div[i]). Odd column 2i+1: cos(pos / div[i]). Angles are in radians.",
  "X": "Add Es and PE entry by entry (same row, same column).",
  "S1s": "Divide every score by sqrt(d_k) = sqrt(2) ≈ 1.4142.",
  "S2s": "Divide every score by sqrt(d_k) = sqrt(2) ≈ 1.4142.",
  "A1": "For each row: take e**x of every entry, then divide each by the row's total. Each row must sum to 1.",
  "A2": "For each row: take e**x of every entry, then divide each by the row's total. Each row must sum to 1.",
  "C": "Put H1 and H2 side by side: the first d_k columns are H1, the next d_k are H2.",
  "R1": "Residual: add the layer's input X to MHA, entry by entry.",
  "mu1": "The mean of each row of R1 (sum the 4 numbers, divide by 4).",
  "sd1": "For each row: average the squared distances from the row's mean, then take the square root.",
  "X1": "(R1[r, c] - mu1[r]) / sd1[r]. Every row ends up with mean 0 and spread 1.",
  "P": "X1 @ W1, then add b1[c] to every entry in column c.",
  "Hf": "ReLU: keep positive numbers, replace negatives with 0.",
  "F": "Hf @ W2, then add b2[c] to every entry in column c.",
  "R2": "Residual again: X1 + F, entry by entry.",
  "mu2": "The mean of each row of R2.",
  "sd2": "The standard deviation of each row of R2 (same recipe as sd1).",
  "Z": "(R2[r, c] - mu2[r]) / sd2[r].",
}
_MATMUL = {"Q1": ("X", W_Q1, "W_Q1"), "K1": ("X", W_K1, "W_K1"), "V1": ("X", W_V1, "W_V1"),
           "Q2": ("X", W_Q2, "W_Q2"), "K2": ("X", W_K2, "W_K2"), "V2": ("X", W_V2, "W_V2"),
           "S1": ("Q1", None, "K1ᵀ"), "S2": ("Q2", None, "K2ᵀ"),
           "H1": ("A1", None, "V1"), "H2": ("A2", None, "V2"), "MHA": ("C", W_O, "W_O")}

def _hint(name, r, c):
    if name in _MATMUL:
        a, B, bname = _MATMUL[name]
        A = got(a)
        if B is None:
            B = {"S1": lambda: got("K1").T, "S2": lambda: got("K2").T,
                 "H1": lambda: got("V1"), "H2": lambda: got("V2")}[name]()
        terms = " + ".join(f"({A[r, j]:.2f})×({B[j, c]:.2f})" for j in range(A.shape[1]))
        return f"{name}[{r},{c}] = row {r} of {a} · column {c} of {bname} = {terms}"
    return _HINTS.get(name, "")

def check(name, yours):
    """Grade your matrix. Wrong or blank cells are listed with a hint for the first one."""
    exp = _RULES[name]()
    y = np.asarray(yours, dtype=float)
    if y.shape != exp.shape:
        print(f"✗ {name}: shape {y.shape}, but it should be {exp.shape}."); return
    if np.isnan(y).all():
        print(f"… {name} is still blank. Replace every nan with your number."); return
    tol = 0.02 + 0.01 * np.abs(exp)
    bad = np.argwhere(np.isnan(y) | (np.abs(y - exp) > tol))
    if len(bad) == 0:
        _ok[name] = y
        print(f"✓ {name} {exp.shape} is correct. Nice work!"); return
    print(f"✗ {name}: {len(bad)} of {exp.size} entries need another look:")
    for idx in bad[:6]:
        where = f"[{', '.join(map(str, idx))}]"
        print(f"   {name}{where}: you have {y[tuple(idx)]:.4g}")
    idx = bad[0]
    r, c = (int(idx[0]), int(idx[1])) if y.ndim == 2 else (int(idx[0]), 0)
    print("   Hint:", _hint(name, r, c))
    print(f"   Stuck? reveal('{name}') shows the answer and lets you carry on.")

def reveal(name):
    """Show the answer for one matrix and use it from here on."""
    exp = _RULES[name](); _ok[name] = exp
    print(f"{name} {exp.shape} =\n{np.round(exp, 4)}")

def check_shapes(answers):
    """Grade a dictionary of shapes for the paper's sizes."""
    PAPER = {"E": (6, 512), "W_Q1": (512, 64), "Q1": (6, 64), "S1": (6, 6), "A1": (6, 6),
             "H1": (6, 64), "C": (6, 512), "W_O": (512, 512), "P": (6, 2048), "Z": (6, 512)}
    right = 0
    for k, v in PAPER.items():
        mine = tuple(answers.get(k, ()))
        ok = mine == v; right += ok
        print(f"{'✓' if ok else '✗'} {k:5s} you said {mine or '(blank)'}" + ("" if ok else f", the paper has {v}"))
    print(f"\n{right} of {len(PAPER)} right.")

print("Ready. Sentence:", sentence.split(), "| n =", n, "| d_model =", d_model, "| h =", h, "| d_k =", d_k, "| d_ff =", d_ff)

In [ ]:
# Run this cell before the decoder part. It adds the German words, the decoder's
# weights and the decoder's answer key to the checker. Again, no peeking at REF.

# ---- the shared vocabulary grows: the paper uses ONE vocabulary for both languages (§3.4) ----
vocab.update({"<s>": 6, "</s>": 7, "die": 8, "katze": 9, "saß": 10})
if len(W_emb) == 6:   # (so rerunning this cell is safe)
  W_emb = np.vstack([W_emb, np.array([
    [ 0.2,  0.3, -0.4, -0.3],   # 6 <s>   start token
    [-0.1,  0.1,  0.3, -0.2],   # 7 </s>  end token
    [-0.4, -0.3, -0.1, -0.1],   # 8 die
    [ 0.4, -0.1,  0.2,  0.1],   # 9 katze
    [-0.4,  0.2, -0.4, -0.4],   # 10 saß
  ])])
words = {i: w for w, i in vocab.items()}
V = len(vocab)   # 11 here, about 37,000 in the paper

# ---- the translation, shifted right ----
target    = "die katze saß"
dec_input = ["<s>", "die", "katze", "saß"]     # what the decoder reads
dec_label = ["die", "katze", "saß", "</s>"]    # what it must predict at each position
m = len(dec_input)                            # 4 positions (the source has n = 3)
inf = np.inf

# ---- masked self-attention weights (the decoder's own; not shared with the encoder) ----
W_mQ1 = np.array([[ 0.0, -0.5], [ 0.0,  0.0], [-1.0, -1.0], [ 0.5,  1.0]])
W_mK1 = np.array([[ 0.5,  0.5], [ 0.0, -1.0], [ 0.0,  0.0], [ 0.5,  0.0]])
W_mV1 = np.array([[ 0.0, -1.0], [ 0.5, -0.5], [ 0.0, -1.0], [ 0.0,  0.0]])
W_mQ2 = np.array([[ 0.0,  0.0], [-1.0,  0.0], [ 0.5,  1.0], [-0.5,  0.5]])
W_mK2 = np.array([[ 0.0,  0.0], [ 0.5, -0.5], [-0.5,  0.0], [ 0.0,  1.0]])
W_mV2 = np.array([[-0.5,  0.0], [ 1.0,  1.0], [ 0.5,  0.0], [ 0.5, -0.5]])
W_mO  = np.array([[ 0.0, -0.5,  0.0, -0.5],
                  [ 0.5, -0.5,  0.0,  0.0],
                  [ 0.5,  0.0, -0.5,  0.0],
                  [-0.5,  0.5,  0.5,  0.0]])

# ---- cross-attention weights: W_xQ is applied to the decoder, W_xK and W_xV to Z ----
W_xQ1 = np.array([[-1.0,  0.0], [-0.5,  0.0], [-0.5,  0.0], [ 0.5, -0.5]])
W_xK1 = np.array([[ 0.0,  0.0], [-0.5,  0.0], [ 0.5,  0.0], [ 0.5,  0.0]])
W_xV1 = np.array([[ 1.0, -1.0], [ 0.0,  1.0], [ 0.0, -1.0], [ 1.0, -0.5]])
W_xQ2 = np.array([[-1.0, -0.5], [ 1.0,  0.0], [-1.0,  0.0], [-1.0,  0.0]])
W_xK2 = np.array([[ 0.0,  0.5], [-1.0,  0.0], [ 0.0,  1.0], [-1.0,  0.0]])
W_xV2 = np.array([[ 0.0,  0.0], [ 0.5,  0.0], [ 0.5, -1.0], [-0.5, -0.5]])
W_xO  = np.array([[ 0.0,  0.0, -0.5,  0.0],
                  [ 0.5, -0.5,  0.0, -0.5],
                  [ 0.5,  0.0,  0.5,  0.5],
                  [ 0.5,  0.0,  0.0,  0.0]])

# ---- the decoder's feed-forward network ----
W1d = np.array([[-0.5,  0.5, -0.5,  0.0,  0.5, -0.5, -0.5,  0.0],
                [-1.0,  0.0, -0.5, -1.0, -1.0, -0.5,  0.0,  0.0],
                [-0.5, -1.0,  0.0,  0.0,  0.0,  0.0, -0.5, -1.0],
                [ 0.0, -0.5, -1.0,  1.0,  0.0,  0.0, -0.5, -0.5]])
b1d = np.array([0.0, 0.1, -0.1, 0.0, -0.1, 0.0, 0.1, 0.1])
W2d = np.array([[ 0.0,  0.0,  0.0, -0.5],
                [-0.5,  0.0, -0.5,  0.5],
                [ 0.5,  0.0,  0.5,  0.0],
                [ 0.0,  0.5,  0.0, -0.5],
                [ 0.0,  0.5,  0.0,  0.0],
                [ 0.0,  0.0,  0.5,  0.5],
                [ 0.0,  0.5,  0.5, -0.5],
                [ 0.0,  0.0,  0.0,  0.0]])
b2d = np.array([-0.1, 0.1, 0.0, -0.1])

MASK = np.triu(np.full((m, m), -np.inf), 1)    # used by the checker only; you build it yourself

# =============== the decoder checker (no need to read below) ===============
_DEC = {
  "ids_d": lambda: np.array([vocab[t] for t in dec_input], dtype=float),
  "Ed":    lambda: W_emb[got("ids_d").astype(int)],
  "Eds":   lambda: got("Ed") * np.sqrt(d_model),
  "PEd":   lambda: np.array([[f(p / got("div")[c // 2]) for c, f in
                              zip(range(d_model), [np.sin, np.cos] * (d_model // 2))] for p in range(m)]),
  "Y0":    lambda: got("Eds") + got("PEd"),
}
for _h in "12":
    _DEC.update({
      f"mQ{_h}": (lambda h: lambda: got("Y0") @ globals()[f"W_mQ{h}"])(_h),
      f"mK{_h}": (lambda h: lambda: got("Y0") @ globals()[f"W_mK{h}"])(_h),
      f"mV{_h}": (lambda h: lambda: got("Y0") @ globals()[f"W_mV{h}"])(_h),
      f"mS{_h}": (lambda h: lambda: got(f"mQ{h}") @ got(f"mK{h}").T)(_h),
      f"mS{_h}s": (lambda h: lambda: got(f"mS{h}") / np.sqrt(d_k))(_h),
      f"mS{_h}m": (lambda h: lambda: got(f"mS{h}s") + MASK)(_h),
      f"mA{_h}": (lambda h: lambda: _sm(got(f"mS{h}m")))(_h),
      f"mH{_h}": (lambda h: lambda: got(f"mA{h}") @ got(f"mV{h}"))(_h),
    })
_DEC.update({
  "mC":   lambda: np.concatenate([got("mH1"), got("mH2")], axis=1),
  "mMHA": lambda: got("mC") @ W_mO,
  "R1d":  lambda: got("Y0") + got("mMHA"),
  "mu1d": lambda: _ln(got("R1d"))[0], "sd1d": lambda: _ln(got("R1d"))[1],
  "Y1":   lambda: (got("R1d") - got("mu1d")[:, None]) / got("sd1d")[:, None],
})
for _h in "12":
    _DEC.update({
      f"xQ{_h}": (lambda h: lambda: got("Y1") @ globals()[f"W_xQ{h}"])(_h),
      f"xK{_h}": (lambda h: lambda: got("Z") @ globals()[f"W_xK{h}"])(_h),
      f"xV{_h}": (lambda h: lambda: got("Z") @ globals()[f"W_xV{h}"])(_h),
      f"xS{_h}": (lambda h: lambda: got(f"xQ{h}") @ got(f"xK{h}").T)(_h),
      f"xS{_h}s": (lambda h: lambda: got(f"xS{h}") / np.sqrt(d_k))(_h),
      f"xA{_h}": (lambda h: lambda: _sm(got(f"xS{h}s")))(_h),
      f"xH{_h}": (lambda h: lambda: got(f"xA{h}") @ got(f"xV{h}"))(_h),
    })
_DEC.update({
  "xC":    lambda: np.concatenate([got("xH1"), got("xH2")], axis=1),
  "xMHA":  lambda: got("xC") @ W_xO,
  "R2d":   lambda: got("Y1") + got("xMHA"),
  "mu2d":  lambda: _ln(got("R2d"))[0], "sd2d": lambda: _ln(got("R2d"))[1],
  "Y2":    lambda: (got("R2d") - got("mu2d")[:, None]) / got("sd2d")[:, None],
  "Pd":    lambda: got("Y2") @ W1d + b1d,
  "Hd":    lambda: np.maximum(0, got("Pd")),
  "Fd":    lambda: got("Hd") @ W2d + b2d,
  "R3d":   lambda: got("Y2") + got("Fd"),
  "mu3d":  lambda: _ln(got("R3d"))[0], "sd3d": lambda: _ln(got("R3d"))[1],
  "Yout":  lambda: (got("R3d") - got("mu3d")[:, None]) / got("sd3d")[:, None],
  "logits": lambda: got("Yout") @ W_emb.T,
  "probs": lambda: _sm(got("logits")),
  "pred":  lambda: got("probs").argmax(axis=1).astype(float),
  "nll":   lambda: -np.log(got("probs")[np.arange(m), [vocab[w] for w in dec_label]]),
})
_RULES.update(_DEC)
for _k, _f in _DEC.items():
    REF[_k] = _f()

_HINTS.update({
  "ids_d": "Look up each decoder INPUT token (⟨s⟩ first), not the words it has to predict.",
  "Ed": "Same lookup as the encoder: row r = W_emb[ids_d[r]].",
  "Eds": "Multiply every entry by sqrt(d_model) = 2, exactly like the encoder.",
  "PEd": "Same formula as the encoder's PE. Rows 0 to 2 are identical to PE; only row 3 (position 3) is new.",
  "Y0": "Eds + PEd, entry by entry.",
  "mS1s": "Divide every score by sqrt(2) ≈ 1.4142.", "mS2s": "Divide every score by sqrt(2) ≈ 1.4142.",
  "mS1m": "Keep entries with column ≤ row. Every entry ABOVE the diagonal (column > row) becomes -inf.",
  "mS2m": "Keep entries with column ≤ row. Every entry ABOVE the diagonal (column > row) becomes -inf.",
  "mA1": "Softmax each row. e**(-inf) = 0, so masked entries get exactly 0 weight; divide by the total of the rest.",
  "mA2": "Softmax each row. e**(-inf) = 0, so masked entries get exactly 0 weight; divide by the total of the rest.",
  "mC": "Put mH1 and mH2 side by side.",
  "R1d": "Residual: Y0 + mMHA.", "mu1d": "Mean of each row of R1d.", "sd1d": "Standard deviation of each row of R1d (divide by 4).",
  "Y1": "(R1d[r, c] - mu1d[r]) / sd1d[r].",
  "xS1s": "Divide every score by sqrt(2).", "xS2s": "Divide every score by sqrt(2).",
  "xA1": "Softmax each row over the 3 ENGLISH words. No mask: the whole source sentence is allowed.",
  "xA2": "Softmax each row over the 3 ENGLISH words. No mask: the whole source sentence is allowed.",
  "xC": "Put xH1 and xH2 side by side.",
  "R2d": "Residual: Y1 + xMHA.", "mu2d": "Mean of each row of R2d.", "sd2d": "Standard deviation of each row of R2d.",
  "Y2": "(R2d[r, c] - mu2d[r]) / sd2d[r].",
  "Pd": "Y2 @ W1d, then add b1d[c] to column c.", "Hd": "ReLU: negatives become 0.",
  "Fd": "Hd @ W2d, then add b2d[c] to column c.",
  "R3d": "Residual: Y2 + Fd.", "mu3d": "Mean of each row of R3d.", "sd3d": "Standard deviation of each row of R3d.",
  "Yout": "(R3d[r, c] - mu3d[r]) / sd3d[r].",
  "logits": "logits[r, w] = Yout[r] · W_emb[w]: the decoder's vector dotted with word w's embedding row.",
  "probs": "Softmax each row of logits over all V words.",
  "pred": "For each row of probs, the ID (column number) of the biggest probability.",
  "nll": "nll[r] = -ln(probs[r, id of dec_label[r]]): minus the log of the probability given to the CORRECT next word.",
})
_DEC_MM = {}
for _h in "12":
    _DEC_MM.update({
      f"mQ{_h}": ("Y0", (lambda h: lambda: globals()[f"W_mQ{h}"])(_h), f"W_mQ{_h}"),
      f"mK{_h}": ("Y0", (lambda h: lambda: globals()[f"W_mK{h}"])(_h), f"W_mK{_h}"),
      f"mV{_h}": ("Y0", (lambda h: lambda: globals()[f"W_mV{h}"])(_h), f"W_mV{_h}"),
      f"mS{_h}": (f"mQ{_h}", (lambda h: lambda: got(f"mK{h}").T)(_h), f"mK{_h}ᵀ"),
      f"mH{_h}": (f"mA{_h}", (lambda h: lambda: got(f"mV{h}"))(_h), f"mV{_h}"),
      f"xQ{_h}": ("Y1", (lambda h: lambda: globals()[f"W_xQ{h}"])(_h), f"W_xQ{_h}"),
      f"xK{_h}": ("Z", (lambda h: lambda: globals()[f"W_xK{h}"])(_h), f"W_xK{_h}"),
      f"xV{_h}": ("Z", (lambda h: lambda: globals()[f"W_xV{h}"])(_h), f"W_xV{_h}"),
      f"xS{_h}": (f"xQ{_h}", (lambda h: lambda: got(f"xK{h}").T)(_h), f"xK{_h}ᵀ"),
      f"xH{_h}": (f"xA{_h}", (lambda h: lambda: got(f"xV{h}"))(_h), f"xV{_h}"),
    })
_DEC_MM.update({"mMHA": ("mC", lambda: W_mO, "W_mO"), "xMHA": ("xC", lambda: W_xO, "W_xO"),
                "logits": ("Yout", lambda: W_emb.T, "W_embᵀ")})
_enc_hint = globals().get('_enc_hint', _hint)
def _hint(name, r, c):
    if name in _DEC_MM:
        a, Bf, bname = _DEC_MM[name]
        A, B = got(a), Bf()
        terms = " + ".join(f"({A[r, j]:.2f})×({B[j, c]:.2f})" for j in range(A.shape[1]))
        return f"{name}[{r},{c}] = row {r} of {a} · column {c} of {bname} = {terms}"
    return _enc_hint(name, r, c)

def check(name, yours):
    """Grade your matrix. Wrong or blank cells are listed with a hint for the first one."""
    exp = _RULES[name]()
    y = np.asarray(yours, dtype=float)
    if y.shape != exp.shape:
        print(f"✗ {name}: shape {y.shape}, but it should be {exp.shape}."); return
    if np.isnan(y).all():
        print(f"… {name} is still blank. Replace every nan with your number."); return
    fin = np.isfinite(exp)
    tol = 0.02 + 0.01 * np.abs(np.where(fin, exp, 0))
    with np.errstate(invalid="ignore"):
        good = np.where(fin, np.abs(y - np.where(fin, exp, 0)) <= tol, y == exp)
    if name == "pred":
        good = y == exp
    bad = np.argwhere(np.isnan(y) | ~good)
    if len(bad) == 0:
        _ok[name] = y
        print(f"✓ {name} {exp.shape} is correct. Nice work!"); return
    print(f"✗ {name}: {len(bad)} of {exp.size} entries need another look:")
    for idx in bad[:6]:
        where = f"[{', '.join(map(str, idx))}]"
        print(f"   {name}{where}: you have {y[tuple(idx)]:.4g}")
    idx = bad[0]
    r, c = (int(idx[0]), int(idx[1])) if y.ndim == 2 else (int(idx[0]), 0)
    print("   Hint:", _hint(name, r, c))
    print(f"   Stuck? reveal('{name}') shows the answer and lets you carry on.")

def check_dec_shapes(answers):
    """Grade a dictionary of decoder shapes for the paper's sizes (6 English tokens, 7 decoder positions)."""
    PAPER = {"Y0": (7, 512), "mS1": (7, 7), "mA1": (7, 7), "xQ1": (7, 64), "xK1": (6, 64),
             "xS1": (7, 6), "xH1": (7, 64), "Pd": (7, 2048), "Yout": (7, 512), "logits": (7, 37000)}
    right = 0
    for k, v in PAPER.items():
        mine = tuple(answers.get(k, ()))
        ok = mine == v; right += ok
        print(f"{'✓' if ok else '✗'} {k:6s} you said {mine or '(blank)'}" + ("" if ok else f", the paper has {v}"))
    print(f"\n{right} of {len(PAPER)} right.")

print("Decoder ready. Source:", sentence.split(), "→ decoder input:", dec_input, "| m =", m, "| V =", V)

In [ ]:
# Run this cell third. It adds the decoder as a function you can call on any list of tokens,
# and the checker for this notebook. Please don't peek at the rules: they are the answer key.

def got(k):   # your checked answer if you have one, otherwise the answer key
    return _ok[k] if k in _ok else REF[k] if k in REF else _RULES[k]()

MAX_TURNS = 4   # stop here if </s> never comes (paper: source length + 50, §6.1)
Zf = REF["Z"]    # the encoder output for "the cat sat", computed ONCE and reused every turn

def _pe(pos):
    return np.array([f(pos / 10000 ** (2 * (c // 2) / d_model)) for c, f in zip(range(d_model), [np.sin, np.cos] * (d_model // 2))])
def _y0(tokens):
    return np.array([W_emb[vocab[t]] * np.sqrt(d_model) + _pe(p) for p, t in enumerate(tokens)])
def _mha(Xq, Xkv, Ws, WO, masked):
    outs = []
    for WQ, WK, WV in Ws:
        S = (Xq @ WQ) @ (Xkv @ WK).T / np.sqrt(d_k)
        if masked:
            S = S + np.triu(np.full(S.shape, -np.inf), 1)
        outs.append(_sm(S) @ (Xkv @ WV))
    return np.concatenate(outs, axis=1) @ WO
def _norm(R): return _ln(R)[2]

def decoder_output(tokens):
    """The whole decoder layer on a list of tokens (you built every piece of it in the last notebook)."""
    Y0 = _y0(tokens)
    Y1 = _norm(Y0 + _mha(Y0, Y0, [(W_mQ1, W_mK1, W_mV1), (W_mQ2, W_mK2, W_mV2)], W_mO, True))
    Y2 = _norm(Y1 + _mha(Y1, Zf, [(W_xQ1, W_xK1, W_xV1), (W_xQ2, W_xK2, W_xV2)], W_xO, False))
    return _norm(Y2 + np.maximum(0, Y2 @ W1d + b1d) @ W2d + b2d)

NEVER_WRITTEN = [vocab["<pad>"], vocab["<s>"]]   # the decoder never has to write these two

def last_logits(tokens):
    """Logits of the LAST row only, rounded to 2 decimals so you can work with them by hand.
    <pad> and <s> get -inf: they are never a correct next word, so they are ruled out."""
    L = np.round(decoder_output(tokens)[-1] @ W_emb.T, 2)
    L[NEVER_WRITTEN] = -np.inf
    return L

def tokens_for_turn(t):
    """<s> followed by the words you picked at turns 1 .. t-1."""
    return ["<s>"] + [words[int(got(f"w{k}")[0])] for k in range(1, t)]

def show_logits(t):
    toks = tokens_for_turn(t); L = last_logits(toks)
    print(f"turn {t}: the decoder reads {toks}")
    print("last-row logits:")
    for i in range(V): print(f"   {i:2d} {words[i]:6s} {L[i]:6.2f}")
    return L

# ---- the answer key, as rules that use YOUR earlier checked answers ----
def _greedy_rules(t):
    return {f"p{t}": lambda: _sm(last_logits(tokens_for_turn(t))[None, :])[0],
            f"w{t}": lambda: np.array([float(np.argmax(got(f"p{t}")))])}
for _t in range(1, MAX_TURNS + 1):
    _RULES.update(_greedy_rules(_t))

def _safe_log(P): return np.log(P, where=P > 0, out=np.full(P.shape, -np.inf))
_new = lambda: words[int(got("w1")[0])]          # the word you picked at turn 1
_RULES.update({
  "k_cache": lambda: _y0(["<s>"]) @ W_mK1,      # given: the cache after turn 1
  "v_cache": lambda: _y0(["<s>"]) @ W_mV1,
  "y0_new": lambda: (W_emb[vocab[_new()]] * np.sqrt(d_model) + _pe(1))[None, :],
  "q_new":  lambda: got("y0_new") @ W_mQ1,
  "k_new":  lambda: got("y0_new") @ W_mK1,
  "v_new":  lambda: got("y0_new") @ W_mV1,
  "K_cache": lambda: np.vstack([got("k_cache"), got("k_new")]),
  "V_cache": lambda: np.vstack([got("v_cache"), got("v_new")]),
  "s_new":  lambda: got("q_new") @ got("K_cache").T / np.sqrt(d_k),
  "a_new":  lambda: _sm(got("s_new")),
  "h_new":  lambda: got("a_new") @ got("V_cache"),
  "beam1_ids":    lambda: np.argsort(-got("p1"))[:2].astype(float),
  "beam1_scores": lambda: np.log(got("p1")[got("beam1_ids").astype(int)]),
  "cand": lambda: got("beam1_scores")[:, None] + _safe_log(np.vstack([
              _sm(last_logits(["<s>", words[int(i)]])[None, :])[0] for i in got("beam1_ids")])),
  "beam2": lambda: np.array(sorted([divmod(int(k), V) for k in np.argsort(-got("cand").ravel())[:2]],
              key=lambda rc: -got("cand")[rc]), dtype=float),
  "lp": lambda: np.array([((5 + L) / 6) ** 0.6 for L in (1, 2, 3, 4)]),
})
_EXACT = {f"w{t}" for t in range(1, MAX_TURNS + 1)} | {"beam1_ids", "beam2"}
_HINTS.update({f"p{t}": "Softmax: e**logit for every word (e**-inf = 0), then divide each by their total. The 11 numbers add up to 1." for t in range(1, MAX_TURNS + 1)})
_HINTS.update({f"w{t}": "Greedy: the ID (0 to 10) of the biggest probability in the row you just checked." for t in range(1, MAX_TURNS + 1)})
_HINTS.update({
  "y0_new": "Same recipe as always: W_emb row of the new word × 2, plus PE for position 1 (sin(1), cos(1), sin(0.01), cos(0.01)).",
  "q_new": "y0_new · W_mQ1: one row times a 4 × 2 matrix.", "k_new": "y0_new · W_mK1.", "v_new": "y0_new · W_mV1.",
  "K_cache": "Stack them: row 0 is k_cache (from turn 1), row 1 is k_new.", "V_cache": "Row 0 is v_cache, row 1 is v_new.",
  "s_new": "q_new · K_cacheᵀ, then divide by sqrt(2). One score per cached token.",
  "a_new": "Softmax of the 2 scores. No mask needed: both cached tokens are in the past.",
  "h_new": "a_new · V_cache: a weighted mix of the 2 cached value rows.",
  "beam1_ids": "The IDs of the two biggest probabilities in p1, biggest first.",
  "beam1_scores": "ln of their probabilities (np.log is fine as a calculator).",
  "cand": "cand[b, w] = beam1_scores[b] + ln P(word w | <s>, beam b's word).",
  "beam2": "Find the two biggest entries of cand; write each as [row, column], biggest first.",
  "lp": "((5 + L) / 6) ** 0.6 for L = 1, 2, 3, 4.",
})

def check(name, yours):
    """Grade your answer. Wrong or blank entries are listed with a hint."""
    exp = np.asarray(_RULES[name](), dtype=float)
    y = np.asarray(yours, dtype=float)
    if y.shape != exp.shape:
        print(f"✗ {name}: shape {y.shape}, but it should be {exp.shape}."); return
    if np.isnan(y).all():
        print(f"… {name} is still blank. Replace every nan with your number."); return
    with np.errstate(invalid="ignore"):
        good = (y == exp) if name in _EXACT else (y == exp) | (np.abs(y - exp) <= 0.02 + 0.01 * np.abs(np.where(np.isfinite(exp), exp, 0)))
    bad = np.argwhere(np.isnan(y) | ~good)
    if len(bad) == 0:
        _ok[name] = y
        print(f"✓ {name} {exp.shape} is correct. Nice work!"); return
    print(f"✗ {name}: {len(bad)} of {exp.size} entries need another look:")
    for idx in bad[:6]:
        print(f"   {name}[{', '.join(map(str, idx))}]: you have {y[tuple(idx)]:.4g}")
    print("   Hint:", _HINTS.get(name, ""))
    print(f"   Stuck? reveal('{name}') shows the answer and lets you carry on.")

def reveal(name):
    """Show the answer and use it from here on."""
    exp = np.asarray(_RULES[name](), dtype=float); _ok[name] = exp
    print(f"{name} {exp.shape} =\n{np.round(exp, 4)}")

print("Inference ready. Z is", Zf.shape, "and stays fixed. Vocabulary:", words)

---

# Part 1: greedy decoding

The encoder has already run, once. `Zf` holds its output, and it won't change again however many words the decoder writes.

The decoder starts from `<s>` alone. Each turn:

1. run the decoder on everything written so far and keep **only the last row**,
2. turn its 11 logits into probabilities with a softmax,
3. pick the word with the highest probability (**greedy**),
4. append it and go again, unless it was `</s>`.

## Turn 1

The decoder reads one token, `<s>`. Two logits are `-inf`: `<pad>` and `<s>` are never a correct next word, so they're ruled out, just like the masked scores in the last notebook (e**-inf = 0). Run the cell to see the last row's logits.

**Shapes:** decoder input (1,) tokens → decoder output (1 × 4) → last row (1 × 4) → logits (11,).

In [ ]:
L1 = show_logits(1)

In [ ]:
# softmax of L1: e**L1[i] / (sum of all e**L1), for i = 0 .. 10
p1 = np.array([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan])
check('p1', p1)

In [ ]:
# the ID of the word greedy picks (the biggest probability)
w1 = np.array([nan])
check('w1', w1)

In [ ]:
print('picked:', words[int(got('w1')[0])], '| so far:', tokens_for_turn(2))

## Turn 2

The decoder reads `<s>` and the 1 word you picked so far. Run the cell to see the last row's logits. Only the last row matters: every earlier row predicts a word you've already written.

**Shapes:** decoder input (2,) tokens → decoder output (2 × 4) → last row (1 × 4) → logits (11,).

In [ ]:
L2 = show_logits(2)

In [ ]:
# softmax of L2: e**L2[i] / (sum of all e**L2), for i = 0 .. 10
p2 = np.array([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan])
check('p2', p2)

In [ ]:
# the ID of the word greedy picks (the biggest probability)
w2 = np.array([nan])
check('w2', w2)

In [ ]:
print('picked:', words[int(got('w2')[0])], '| so far:', tokens_for_turn(3))

## Turn 3

The decoder reads `<s>` and the 2 words you picked so far. Run the cell to see the last row's logits.

**Shapes:** decoder input (3,) tokens → decoder output (3 × 4) → last row (1 × 4) → logits (11,).

In [ ]:
L3 = show_logits(3)

In [ ]:
# softmax of L3: e**L3[i] / (sum of all e**L3), for i = 0 .. 10
p3 = np.array([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan])
check('p3', p3)

In [ ]:
# the ID of the word greedy picks (the biggest probability)
w3 = np.array([nan])
check('w3', w3)

In [ ]:
print('picked:', words[int(got('w3')[0])], '| so far:', tokens_for_turn(4))

## Turn 4

The decoder reads `<s>` and the 3 words you picked so far. Run the cell to see the last row's logits.

**Shapes:** decoder input (4,) tokens → decoder output (4 × 4) → last row (1 × 4) → logits (11,).

In [ ]:
L4 = show_logits(4)

In [ ]:
# softmax of L4: e**L4[i] / (sum of all e**L4), for i = 0 .. 10
p4 = np.array([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan])
check('p4', p4)

In [ ]:
# the ID of the word greedy picks (the biggest probability)
w4 = np.array([nan])
check('w4', w4)

In [ ]:
print('picked:', words[int(got('w4')[0])], '| so far:', tokens_for_turn(5))

## How did it end?

`</s>` never came, so the loop stopped at the limit of 4 turns. The paper would allow 3 + 50 = 53. Notice how it starts repeating itself. Greedy decoding loves to get stuck in loops, even in trained models, which is one reason for the length limit and for the alternatives below. With made-up weights the words are nonsense; a trained model would write "die katze saß </s>" through exactly these steps.

Notice that the encoder ran **once**, and the decoder ran **4 times**, each time on one more token.

In [ ]:
print('greedy translation:', ' '.join(tokens_for_turn(4)[1:] + [words[int(got('w4')[0])]]))

---

# Part 2: the KV cache

At turn 2 the decoder recomputed the `<s>` row in every sublayer, and got exactly the same numbers as at turn 1: the mask means `<s>` can never see the new word. A **KV cache** keeps each layer's keys and values instead, so each turn only the **new token's row** is computed.

Let's do that for turn 2, in the first masked self-attention head. The cache after turn 1 holds one row each:

In [ ]:
k_cache = _RULES['k_cache'](); v_cache = _RULES['v_cache']()
_ok['k_cache'] = k_cache; _ok['v_cache'] = v_cache
print('k_cache (1 × 2) =', np.round(k_cache, 4))
print('v_cache (1 × 2) =', np.round(v_cache, 4))
print('new token:', words[int(got('w1')[0])], 'at position 1')
print('W_mQ1 =\n', W_mQ1, '\nW_mK1 =\n', W_mK1, '\nW_mV1 =\n', W_mV1)

### The new token's input row

Embedding × √4 = 2, plus positional encoding for **position 1**. **Shape:** (1 × 4).

In [ ]:
# W_emb[new word] * 2 + PE[1]
y0_new = np.array([
    [nan, nan, nan, nan]
])
check('y0_new', y0_new)

### Its query, key and value

One row each. **Shape:** (1 × 2).

In [ ]:
# y0_new · W_mQ1
q_new = np.array([
    [nan, nan]
])
check('q_new', q_new)

In [ ]:
# y0_new · W_mK1
k_new = np.array([
    [nan, nan]
])
check('k_new', k_new)

In [ ]:
# y0_new · W_mV1
v_new = np.array([
    [nan, nan]
])
check('v_new', v_new)

### Append to the cache

The cache grows by one row. **Shape:** (2 × 2).

In [ ]:
# row 0 = k_cache, row 1 = k_new
K_cache = np.array([
    [nan, nan],
    [nan, nan]
])
check('K_cache', K_cache)

In [ ]:
# row 0 = v_cache, row 1 = v_new
V_cache = np.array([
    [nan, nan],
    [nan, nan]
])
check('V_cache', V_cache)

### One row of attention

Only the new query is scored, against every cached key: (1 × 2) · (2 × 2)ᵀ → **(1 × 2)**. No mask: everything in the cache is in the past.

In [ ]:
# q_new · K_cacheᵀ / sqrt(2)
s_new = np.array([
    [nan, nan]
])
check('s_new', s_new)

In [ ]:
# softmax of s_new
a_new = np.array([
    [nan, nan]
])
check('a_new', a_new)

In [ ]:
# a_new · V_cache
h_new = np.array([
    [nan, nan]
])
check('h_new', h_new)

### Same answer, less work

Now recompute the head the slow way, on both tokens at once with the mask, and compare its last row with yours.

In [ ]:
toks = tokens_for_turn(2)
Y0f = _y0(toks)
S = (Y0f @ W_mQ1) @ (Y0f @ W_mK1).T / np.sqrt(d_k) + np.triu(np.full((2, 2), -np.inf), 1)
H_full = _sm(S) @ (Y0f @ W_mV1)
print('full recompute, both rows:\n', np.round(H_full, 4))
print('your cached row          :', np.round(got('h_new'), 4))
print('match:', np.allclose(H_full[-1], got('h_new'), atol=0.03))

Row 0 of the full recompute is the `<s>` row from turn 1, unchanged. With a cache you never compute it again. Over T turns that's T rows per layer instead of 1 + 2 + … + T.

---

# Part 3: beam search

Greedy keeps one candidate. Beam search keeps the **k best** (here k = 2; the paper uses 4), scoring each by the **sum of the natural logs** of its words' probabilities.

### Turn 1

Start from your `p1`. The two most likely first words become the two beams.

In [ ]:
print('p1 =', np.round(got('p1'), 4))
print(words)

In [ ]:
# IDs of the 2 most likely first words, biggest first
beam1_ids = np.array([nan, nan])
check('beam1_ids', beam1_ids)

In [ ]:
# ln of their probabilities
beam1_scores = np.array([nan, nan])
check('beam1_scores', beam1_scores)

### Turn 2

Each beam is extended by every one of the 11 words: 2 × 11 = 22 candidates. A candidate's score is its beam's score plus ln P(the new word). Here are both beams' probabilities for the second word:

In [ ]:
P2b = np.vstack([_sm(last_logits(["<s>", words[int(i)]])[None, :])[0] for i in got("beam1_ids")])
for b, i in enumerate(got("beam1_ids")):
    print(f"beam {b}: <s> {words[int(i)]}")
    print("   P(next) =", np.round(P2b[b], 3))

In [ ]:
# cand[b, w] = beam1_scores[b] + ln P2b[b, w]; ln 0 = -inf
cand = np.array([
    [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan],
    [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]
])
check('cand', cand)

In [ ]:
# the 2 best candidates as [beam row, word ID], best first
beam2 = np.array([
    [nan, nan],
    [nan, nan]
])
check('beam2', beam2)

In [ ]:
for b, w in got('beam2').astype(int):
    print('kept: <s>', words[int(got('beam1_ids')[b])], words[w], '| score', round(float(got('cand')[b, w]), 3))

### The length penalty

Every extra word adds a negative log, so longer translations always score lower. Finished translations are compared after dividing by lp = ((5 + length) / 6)<sup>α</sup>, with α = 0.6 (§6.1, from Wu et al., 2016). Compute it for lengths 1, 2, 3 and 4.

In [ ]:
# ((5 + L) / 6) ** 0.6 for L = 1, 2, 3, 4
lp = np.array([nan, nan, nan, nan])
check('lp', lp)

## Want more?

- **Finish the beam.** Keep extending both beams until each ends in `</s>`, divide each total by its length penalty, and compare the winner with your greedy translation.
- **Sample instead.** Divide `L1` by a temperature of 2 before the softmax and see how the probabilities flatten. Then draw a word with `np.random.choice(V, p=...)`.
- **Make the weights your own.** Edit the decoder's weights in the second setup cell and rerun: the checker recomputes every answer.
- **Compare with the site.** Part 3 of Attention by Hand runs this loop live, with the KV cache and beam search animated.